# CS 789 — Assignment 6: Build a SOC Copilot
**AI-Powered Cyber Attacks and Defenses · Fall 2026 · UNLV · Prof. Yoohwan Kim**

In A4 the retriever was a list of chunks, in A5 a graph an LLM extracted for you. In A6 you build the core of a
**product**: an AI copilot that answers SOC analysts' questions from a knowledge graph of real IDS/SIEM alerts,
follows a guardrailed policy, and remembers the conversation — and you launch it as a web app with dashboards.
**A7 builds on it**: n8n automation, an uncensored model, and your own dashboards.

| Part | What you build | What you learn |
|---|---|---|
| A | Complete a **SOC ontology** (OWL/Turtle) and a **SHACL** shape | the vocabulary that drives the KG, the extractor and access control |
| B | Load 37,808 alerts into a **LlamaIndex** property graph; extract facts from shift notes with an LLM | deterministic vs. LLM ingestion, entity resolution |
| C | A KG retriever under a **token budget**, a **guardrailed system prompt**, and **3 LLMs** compared | context-window management, guardrails, model choice |
| D | **LangChain memory** in a **SQLite** conversation store with your own history policy | how much history, how to load it, what it costs |
| E | **Launch the SOC Copilot** web app with your chat, and run its acceptance test | from notebook functions to a product |

Data: the public **AIT Alert Data Set** (Landauer, Skopik, Wurzenberger, Zenodo 2023, doi:10.5281/zenodo.8263181,
CC BY 4.0) — Wazuh and Suricata alerts from a simulated enterprise network, scenario *russellmitchell*, reduced to
summary fields. The host inventory, staff list and roles are an invented overlay for this lab.

As in A4/A5: 🛠️ cells you **build with your browser AI assistant** (prompts provided — adapt them), ✅ check cells
tell you whether your code meets its contract, ✍️ questions are yours to answer.

> Runtime: **Colab Pro, L4 GPU** (A100 also fine). About 2½–3 hours of your time, in two sittings.
> Everything runs inside your Colab VM: the models (vLLM), the KG and the web app. No paid API needed.

---
## Part 0 — Setup (≈ 15 min)
`Runtime → Change runtime type → L4 GPU → Save`. Run 0.1: it clones the repository, installs the packages,
installs vLLM in its own environment and starts **Qwen3-4B-Instruct-2507** on port 8000 (about 6 minutes the
first time). If you come back after a runtime reset, just run 0.1 again — it reuses what it can.

In [ ]:
# 0.1 Files, packages, the local model server, and the data
import os, sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/A6"):
        !git clone -q https://github.com/unlv-cs789-f26/A6.git /content/A6
    %cd /content/A6
    if importlib.util.find_spec("llama_index") is None:
        !pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())
import soc6 as H
import pandas as pd
pd.set_option("display.max_colwidth", 120)
H.install_vllm()
H.ensure_model("qwen")
print(H.chat("qwen", [{"role": "user", "content": "Which ATT&CK technique ID is 'Valid Accounts'? One line."}])["text"])
A, hosts, staff, windows = H.load_all()
techniques = pd.read_csv(H.HERE / "data" / "attack_techniques.csv")
H.overview(A)

✍️ **Deliverable 0**: screenshot of the output of 0.1 (the model's one-line answer — is it right? It is T1078 —
and the alerts-per-host-per-day table). Then look at the data yourself:

In [ ]:
display(hosts); display(staff.drop(columns=["phone", "on_call"]))
display(A.sample(8, random_state=6)[["ts", "host", "rule_id", "level", "description", "mitre_ids", "src_ip", "dst_ip"]])
A.groupby(["level"]).size().to_frame("alerts").T

---
## Part A — The ontology (≈ 25 min, no AI for the first attempt)

The ontology is the contract for everything after it: **every class becomes a node label** in the graph
(`AlertGroup` → `ALERT_GROUP`), **every object property a relationship type** (`onHost` → `ON_HOST`), every
datatype property a node property, and `soc:sensitivity "restricted"` tells the access-control layer what a tier-1
analyst must never see. `H.schema_from_ontology()` reads the file; everything after it (and A7) uses what it returns.

Open **`ontology/soc.ttl`** in Colab's file browser (📁 on the left → `A6/ontology/soc.ttl`, double-click to edit)
and do **TODO 1–3**. Then open **`ontology/shapes.ttl`** and do **TODO 4** (a SHACL shape — lecture 7-1).
Save the files (Ctrl+S) and run the check. Write it yourself first — Turtle is short and the check tells you
exactly what is missing. Use the assistant only to explain an error message.

In [ ]:
# A.1 Check your ontology (re-run after every edit)
H.check_ontology()

In [ ]:
# A.2 Look at it: classes are boxes, object properties arrows; restricted items are red (hover for properties)
H.show_ontology()
S = H.schema(quiet=False)
print("node labels :", S["entity_labels"]); print("relations   :", S["relation_labels"]); print("restricted  :", S["restricted"])

✍️ **Questions A-1 … A-3**
* **A-1** The KG stores **alert groups** (one per rule × host × source IP × clock hour), not the 37,808 alerts.
  How many groups are there (`len(H.alert_groups(A))`)? What does an analyst *lose* by grouping, and where would
  you keep the raw alerts instead?
* **A-2** OWL cannot put a property on an edge, so "this group overlaps that exercise window" became a class
  (`ExerciseWindow`) with start/end and an `overlaps` property. Sketch (in Turtle or a diagram) how you would model
  *"IP 172.19.131.174 belonged to host vpn from t1 to t2"* in a **temporal KG**. Why does a SOC need that?
* **A-3** Paste your `HostShape` and the violation messages it produced for the bad host. Propose one more
  constraint SHACL should enforce on `AlertGroup` (write it).

---
## Part B — The knowledge graph with LlamaIndex (≈ 35 min)

Two kinds of ingestion. **Structured** data (alerts, inventory) is mapped *deterministically*: no LLM, every
fact exact. **Unstructured** text (analysts' shift-handover notes) goes through an **LLM extractor** constrained
by the ontology — the facts are useful but not guaranteed. The store is LlamaIndex's
`SimplePropertyGraphStore` (labelled nodes + relations, in memory, saved to JSON). A5 used Neo4j for the same
role; here the KG lives inside the app.

#### 🛠️ B.1 Build the KG from the tables (deterministic)

**Contract** (the checks below depend on these exact names): `kg = build_kg(A, hosts, staff, windows, techniques)` → `SimplePropertyGraphStore`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
I am in a Google Colab notebook. These pandas DataFrames exist:
  A (alerts): alert_id, ts (ISO string), host, rule_id, level (int), description, mitre_ids ("|"-separated
     ATT&CK ids or ""), src_ip, dst_ip, src_user, dst_user, hour (e.g. "2022-01-21T05:00Z"), country
  hosts: host, ip, role, zone, owner, criticality (int)
  staff: username, name, team, app_role, phone, on_call
  windows: window_id, phase, start, end (ISO strings)
  techniques: technique_id, name, tactics ("|"-separated)
Using LlamaIndex (from llama_index.core.graph_stores import SimplePropertyGraphStore;
from llama_index.core.graph_stores.types import EntityNode, Relation) write
build_kg(A, hosts, staff, windows, techniques) that returns a SimplePropertyGraphStore with these nodes
(EntityNode(name=..., label=..., properties={...})):
  PERSON name=username; props fullName, team, appRole, phone, onCall
  HOST name=host; props role, zone, criticality (int)
  IP_ADDRESS name=the IP; props internal (ipaddress.ip_address(ip).is_private), country (most common
     non-empty country among alerts with that src_ip, else "")
  TECHNIQUE name=technique_id; props techName.   TACTIC name=tactic name
  RULE name=f"rule:{rule_id}"; props ruleId, ruleLevel (int, level of its first alert), description
  ACCOUNT name=user name, for every non-empty src_user or dst_user
  EXERCISE_WINDOW name=window_id; props phase, windowStart, windowEnd
  ALERT_GROUP one per (host, rule_id, src_ip, hour), with "-" for an empty src_ip;
     name=f"AG|{host}|{rule_id}|{src}|{hour}"; props hour, firstSeen, lastSeen (min/max ts),
     count (int), maxLevel (int), description
and these relations (Relation(label=..., source_id=..., target_id=...)):
  HOST -HAS_IP-> IP_ADDRESS; HOST -OWNED_BY-> PERSON (hosts.owner); TECHNIQUE -IN_TACTIC-> TACTIC;
  RULE -MAPS_TO-> TECHNIQUE (from mitre_ids); ALERT_GROUP -ON_HOST-> HOST, -MATCHES_RULE-> RULE,
  -FROM_IP-> IP_ADDRESS (skip "-"), -TO_IP-> IP_ADDRESS (each distinct non-empty dst_ip of the group),
  -INVOLVES_ACCOUNT-> ACCOUNT (src_user and dst_user), -OVERLAPS-> EXERCISE_WINDOW when
  firstSeen <= window end and lastSeen >= window start.
Use pandas groupby, not a Python loop over the 37,808 rows. Store integers as Python int, not numpy types.
Upsert all nodes first, then all relations. Then run: import time; t=time.time();
kg = build_kg(A, hosts, staff, windows, techniques); print(time.time()-t, "s").
Put the code in the empty cell under B.1 and run it, then run the check cell below it.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: kg = build_kg(A, hosts, staff, windows, techniques) → SimplePropertyGraphStore


In [ ]:
# B.1 check (if it cannot pass, `kg = H.load_reference_kg()` lets you continue -- B.1 then earns no points)
H.check_kg(kg)
H.save_kg(kg, "soc_kg.json")

In [ ]:
# B.2 Explore (provided): sizes by label, and the neighborhood of the VPN gateway
nodes_by_label, rels_by_type = H.kg_stats(kg)
display(pd.concat([nodes_by_label.rename("nodes"), rels_by_type.rename("relations")], axis=1).fillna(0).astype(int))
H.show_kg(kg, "vpn", depth=1, limit=70)

In [ ]:
# Try other centers: an IP, a rule, a technique, an account. Grey dots are alert groups.
H.show_kg(kg, "T1110.001", depth=2, limit=60)

**B.3 Facts from text.** `notes/` holds three shift-handover notes written by analysts. Who *reviewed*,
*escalated* or *blocked* what is not in any log — it only exists in these notes. LlamaIndex's
`SchemaLLMPathExtractor` asks the LLM for triples, but **only** with the labels and relations your ontology
allows (`strict=True` drops everything else). Then each extracted name must be **resolved** to the KG node it
means ("Sam Whitaker" is the PERSON `hunter1`).

**First, write your own note** (manual, ~5 min): create `notes/my_note.md` (📁 → right-click `notes` → New file)
describing a short imaginary shift in the same style: at least **four facts** using these relations — someone
REVIEWED / ESCALATED a host, BLOCKED an IP, an account LOGGED_INTO a host, a host is OWNED_BY a person or HAS_IP
an IP — using hosts, people and IPs that exist in the KG.

#### 🛠️ B.3 Extract triples from the notes with the LLM

**Contract** (the checks below depend on these exact names): `extracted = extract_note_triples(notes, llm, strict=True)` → `{note file: [(subject, RELATION, object), ...]}`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook: S = H.schema() is a dict with S["validation"] (a list of (subject_label, RELATION,
object_label) tuples). NOTE_RELATIONS = ["REVIEWED", "ESCALATED", "BLOCKED", "LOGGED_INTO", "OWNED_BY", "HAS_IP"].
llm = H.llamaindex_llm("qwen") is a LlamaIndex LLM. notes = H.load_notes() is {file name: text}. kg is a
SimplePropertyGraphStore, and H.index(kg).nodes is a dict {node name: (label, properties)}; PERSON nodes have a
"fullName" property.
Write extract_note_triples(notes, llm, strict=True) -> dict {file name: [(subject, RELATION, object), ...]}:
- build llama_index.core.indices.property_graph.SchemaLLMPathExtractor(llm=llm,
  possible_entities=typing.Literal["PERSON", "HOST", "IP_ADDRESS", "ACCOUNT"],
  possible_relations=typing.Literal[tuple(NOTE_RELATIONS)], kg_validation_schema=[the tuples of S["validation"]
  whose relation is in NOTE_RELATIONS], strict=strict, max_triplets_per_chunk=12, num_workers=1)
- run it on one llama_index.core.schema.TextNode(text=..., id_=file name) per note with
  H.run_async(extractor.acall(nodes)) (Colab already runs an event loop, so extractor(nodes) fails); it returns
  the nodes, each with node.metadata["relations"] (a list of Relation with source_id, label, target_id)
- resolve every source_id / target_id to a KG node name: the name itself if it is a KG node; else the PERSON
  whose fullName equals it (case-insensitive); else the lower-cased name if that is a KG node; else keep it
- return the resolved (source, relation label, target) triples, without duplicates
Then run: extracted = extract_note_triples(notes, llm); display(H.score_extraction(extracted)) and print the
triples extracted from my_note.md. Put the code in the empty cell under B.3.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: extracted = extract_note_triples(notes, llm, strict=True) → {note file: [(subject, RELATION, object), ...]}


In [ ]:
# B.4 Merge the extracted facts into the KG, with provenance on every relation (provided)
from llama_index.core.graph_stores.types import EntityNode, Relation
X = H.index(kg)
added = 0
for note, trips in extracted.items():
    for s, r, o in trips:
        if s in X.nodes and o in X.nodes:          # only facts about things the KG already knows
            kg.upsert_relations([Relation(label=r, source_id=s, target_id=o,
                                          properties={"source": f"notes/{note}", "extracted_by": "llm"})])
            added += 1
print(added, "LLM-extracted relations merged;", H.index(kg).edges.rel.value_counts()[["REVIEWED", "ESCALATED"]].to_dict()
      if "REVIEWED" in H.index(kg).edges.rel.values else "")
H.save_kg(kg, "soc_kg.json")

✍️ **Questions B-1 … B-3**
* **B-1** Copy the node/relation table from B.2 and the build time. Which relation type dominates, and why?
* **B-2** For the three provided notes give precision and recall, then go through every **missed** and **wrong**
  triple and classify it: entity resolution, wrong relation, wrong direction, invented fact, or dropped by the schema.
  Re-run B.3 with `strict=False` once — what changes?
* **B-3** Paste your note and its triples. B.4 merged only facts whose both ends already exist in the KG and
  tagged them `extracted_by: llm`. Why is that a good rule for a SOC? What could go wrong if a malicious or
  careless note were merged without it?

---
## Part C — The answer engine: retrieval under a budget, guardrails, three LLMs (≈ 45 min)

The copilot answers from a **context** that your retriever builds from the KG. The context window is the scarce
resource: the alert history is 37,808 rows, the context gets a few hundred tokens. So the retriever links the
entities in the question, pulls their neighborhood, and replaces raw alerts with **aggregates** (totals, by
rule, by day, top sources). Restricted fields are already hidden by the `H.node_lines` / `H.alert_summary`
helpers when `role != "manager"` — the ontology's `soc:sensitivity` marks are what they read.

#### 🛠️ C.1 kg_context: the retriever

**Contract** (the checks below depend on these exact names): `kg_context(question, role='tier1', budget=1200) -> str`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook kg is a LlamaIndex SimplePropertyGraphStore and the harness H provides:
  H.link_entities(question, kg) -> dict with lists "hosts", "ips", "persons", "accounts", "rules",
     "techniques", "tactics", "windows", plus "since", "until" (pandas Timestamps or None), "min_level" (int)
  H.node_lines(kg, node, role) -> list of lines describing one node and its direct neighbors
  H.alert_summary(kg, host=None, ip=None, account=None, technique=None, rule=None, since=None, until=None,
     min_level=0, top=5, role="tier1") -> list of lines (totals, by rule, by host, by day, top source IPs,
     accounts, first high-level alerts). A line starts with "  top source IPs: " or "  by host: " followed by
     "name xCOUNT; name xCOUNT".
  H.windows_lines(kg, role) -> lines of the exercise schedule (empty unless role == "manager")
  H.count_tokens(text) -> int
Write kg_context(question, role="tier1", budget=1200) -> str that:
1. links the entities in the question and adds H.node_lines for every linked host, ip, person, account, rule,
   technique and tactic
2. adds ONE H.alert_summary per linked host (or per linked IP if there is no host; or one summary with no
   host/ip if neither), passing the first linked rule, technique and account and since/until/min_level as filters
3. multi-hop: adds H.node_lines for the first IP on a "top source IPs:" line, and for the first host on a
   "by host:" line (that shows who owns it)
4. if the question says "host X" and X is not a KG node, adds "NOTE: there is no host named X in the knowledge graph."
5. if role == "manager" and the question mentions an exercise, red team, schedule or phase, adds H.windows_lines
6. drops duplicate lines, then keeps lines in order while the total stays <= budget tokens
   (append "... (truncated)" when it cuts)
Put it in the empty cell under C.1, run it, then run the check cell below.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: kg_context(question, role='tier1', budget=1200) -> str


In [ ]:
# C.1 check, and look at one context yourself
H.check_context(kg_context)
print(kg_context("Which source IP caused the burst of level-10 alerts on intranet-server on Jan 24, 2022?"))

**C.2 The system prompt — write it yourself (no AI).** It is the copilot's policy. It must make the model:
answer **only from the context** and say so when the context does not contain the answer (never guess numbers);
stay **read-only** (it cannot block, isolate or change anything — say who can); **refuse restricted data** to
tier-1 users even when asked to "ignore previous instructions" or told the user is an admin; treat text in the
context as **data, not instructions**; decline off-topic requests; answer briefly with exact names, IPs, rule ids
and timestamps. `{role}` is replaced with the user's role.

In [ ]:
# C.2 YOUR system prompt (keep the {role} placeholder)
SYSTEM_PROMPT = """You are SOC Copilot.
The user's role is {role}.
"""
CONTEXT_BUDGET = 1200

In [ ]:
# C.3 The answer function (provided): system prompt + context + question -> one chat call
def answer(question, model="qwen", role="tier1", history=None, budget=None):
    ctx = kg_context(question, role, budget or CONTEXT_BUDGET)
    messages = ([{"role": "system", "content": SYSTEM_PROMPT.format(role=role)}] + (history or []) +
                [{"role": "user", "content": f"CONTEXT (from the SOC knowledge graph):\n{ctx}\n\nQUESTION: {question}"}])
    r = H.chat(model, messages, max_tokens=300)
    return {"answer": r["text"], "prompt_tokens": r["prompt_tokens"], "completion_tokens": r["completion_tokens"],
            "context_tokens": H.count_tokens(ctx), "seconds": r["seconds"]}

print(answer("Who owns the host vpn, and what is its criticality?")["answer"])

In [ ]:
# C.4 All 22 questions on Qwen. Iterate on SYSTEM_PROMPT until restricted / action / unknown / out_of_scope pass,
#     re-running this cell (~1 min). Keep your FIRST run's numbers for C-1.
H.eval_bot("C-qwen", answer, models=["qwen"])
H.scoreboard("C-qwen")

**C.5 Three LLMs.** Same retriever, same prompt, three model families: **Qwen3-4B** (dense transformer),
**IBM Granite-4.0-H-Micro** (3B, hybrid Mamba-2/transformer), and optionally **Gemini** (Google's hosted model).
The two local models share the GPU, so the harness stops one and starts the other (1–2 min per switch; Granite
downloads ~6 GB the first time). For Gemini, create a free key at aistudio.google.com → *Get API key*, and add it
to Colab 🔑 Secrets as `GEMINI_API_KEY` (notebook access on). Skip Gemini if you prefer — then compare two models.

In [ ]:
# C.5 (~6 min) -- remove "gemini" if you have no key
MODELS = ["qwen", "granite"] + (["gemini"] if H.gemini_key() else [])
H.eval_bot("C-models", answer, models=MODELS)
display(H.scoreboard("C-models")); H.plot_scoreboard("C-models")
H.ensure_model("qwen")       # back to Qwen for the rest of the assignment

In [ ]:
# C.6 Context budget sweep: how much context does each question type need? (~3 min)
H.clear_results("C-budget")
for b in [250, 600, 1200, 2500]:
    H.eval_bot("C-budget", lambda q, model, role: answer(q, model, role, budget=b), models=["qwen"],
               label=f"budget={b}", verbose=False)
H.scoreboard("C-budget", by="label")

✍️ **Questions C-1 … C-4**
* **C-1** Paste your final `SYSTEM_PROMPT`. Show one question whose answer changed because of a rule you added
  (before → after).
* **C-2** The C-models scoreboard: accuracy by type, leaks, prompt tokens, seconds per question. Which model would
  you deploy in a SOC, and why? (Consider: accuracy, refusals, latency, where the data goes.)
* **C-3** The budget sweep: where is the knee? Which question types need the most context, and why? Pick
  `CONTEXT_BUDGET` for the product and justify it in tokens *and* dollars at 1,000 questions/day if the model were
  a paid API at $0.30 per million input tokens.
* **C-4** Take one wrong answer from C-models (`H.results("C-models").query("grade != 'correct'")`). Is it a
  **retrieval** failure (the fact is not in `kg_context(...)`) or a **generation** failure (the fact is there and the
  model got it wrong)? Show the evidence.

---
## Part D — Memory: LangChain + SQLite (≈ 30 min)

A copilot has to remember the conversation: *"Who owns **that host**?"* means nothing without the previous turn.
You decide **what to keep** (all, the last K turns, a token budget, or a summary of older turns), **where to
keep it** (a SQLite table, so it survives a restart and can be audited), and **how to load it** into the context
window. One subtlety: the KG retriever only sees the current question, so memory must also be used to **rewrite**
the follow-up into a standalone question *before* retrieval.

#### 🛠️ D.1 chat_turn with LangChain memory

**Contract** (the checks below depend on these exact names): `chat_turn(session_id, user_text, model='qwen', role='tier1', policy=None) -> dict` ; `load_history(session_id, policy)` ; `DB`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook answer(question, model, role, history) exists (history is a list of {"role", "content"}
dicts) and returns a dict with "answer". H.langchain_chat(model) returns a LangChain ChatOpenAI;
H.count_tokens(text) counts tokens. POLICY is a global string.
Using LangChain, write:
- DB = "sqlite:///conversations.db" and get_history(session_id) returning
  langchain_community.chat_message_histories.SQLChatMessageHistory(session_id=session_id, connection=DB)
- load_history(session_id, policy, model="qwen") -> list of LangChain messages, for policy:
  "none" -> []; "all" -> every message; "window-K" -> the last K user/assistant pairs (2*K messages);
  "tokens-N" -> langchain_core.messages.trim_messages(messages, max_tokens=N, strategy="last", start_on="human",
     token_counter=lambda ms: sum(H.count_tokens(m.content) for m in ms));
  "summary-K" -> the last K pairs verbatim, preceded by ONE SystemMessage("Summary of the earlier conversation: ...")
     that the LLM writes from the older messages (keep it short: names, IPs, hosts, numbers); cache it in a dict
     keyed by (session_id, number of older messages) so it is not recomputed every turn
- rewrite(question, past, model) -> str: if past is empty return question; else the LCEL chain
  ChatPromptTemplate.from_messages([("system", "Rewrite the user's follow-up question as a standalone question.
  Replace pronouns and phrases like 'that host', 'it', 'she' with the exact host names, IP addresses or people
  from the conversation. Output only the question."), MessagesPlaceholder("history"),
  ("human", "Follow-up question: {question}")]) | H.langchain_chat(model) | StrOutputParser()
- chat_turn(session_id, user_text, model="qwen", role="tier1", policy=None) -> dict: policy defaults to POLICY;
  past = load_history(session_id, policy, model); standalone = rewrite(user_text, past, model);
  out = answer(standalone, model=model, role=role, history=[{"role": "user" if m.type == "human" else
  ("assistant" if m.type == "ai" else "system"), "content": m.content} for m in past]); store user_text and
  out["answer"] with add_user_message / add_ai_message; set out["rewritten"] = standalone,
  out["history_messages"] = len(past), out["history_tokens"] = token count of past; return out.
Put the code in the empty cell under D.1 and run it.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: chat_turn(session_id, user_text, model='qwen', role='tier1', policy=None) -> dict ; load_history(session_id, policy) ; DB


In [ ]:
# D.2 The 10-turn conversation under five policies (~4 min). Every turn after the first needs memory.
H.clear_results("D-memory")
for pol in ["none", "window-1", "tokens-300", "summary-2", "all"]:
    H.run_dialogue(lambda sid, text, model, role: chat_turn(sid, text, model, role, policy=pol), policy=pol)
H.plot_dialogue("D-memory")
H.results("D-memory").pivot_table(index="label", values=["correct", "prompt_tokens"], aggfunc={"correct": "sum", "prompt_tokens": "sum"})

In [ ]:
# What did the rewriter make of the follow-ups? (one policy)
H.results("D-memory").query("label == 'summary-2'")[["turn", "rewritten", "correct", "answer"]]

**D.3 Look inside the conversation store (no AI).** LangChain's `SQLChatMessageHistory` writes one row per
message into the table `message_store` (`id, session_id, message` — the message is JSON). Write **two SQL queries
yourself**: (1) messages per session, newest session first; (2) the longest stored AI answer (use
`json_extract(message, '$.data.content')` and `length(...)`).

In [ ]:
import sqlite3
con = sqlite3.connect("conversations.db")
pd.read_sql_query("""
-- your query 1
SELECT 1
""", con)

In [ ]:
pd.read_sql_query("""
-- your query 2
SELECT 1
""", con)

✍️ **Questions D-1 … D-3**
* **D-1** The policy table (turns correct, total prompt tokens). Which turns fail under `none` and `window-1`, and
  why does turn 10 ("the very first host we talked about") separate the policies?
* **D-2** Your two SQL queries and results. The store keeps every question an analyst ever asked. What retention
  rule and what access control would you put on this table in a real SOC?
* **D-3** Choose the policy and budget for the product (`POLICY` — it is used in Part E and in A7) and justify it with your
  numbers. What does the rewrite step cost per turn, and when would you skip it?

---
## Part E — Launch the SOC Copilot (≈ 20 min)

Your retriever, system prompt and memory become a web product: one server (port 8080) with SOC dashboards on the
left, **your** copilot on the right, and a knowledge-graph explorer. The dashboard panels are provided this time —
in A7 you add your own, plus an n8n triage pipeline and an uncensored model. The page has a **view as** selector
(tier1 / manager); real sign-in is out of scope, the role is passed to your `chat_turn`.

In [ ]:
# E.1 Start the product with YOUR chat_turn (Part D) and KG (Part B)
H.register(chat_turn=chat_turn, kg=kg, conversations_db=DB, models=["qwen"])
url = H.start_platform()
print("Open the SOC Copilot:", url)

**E.2 Use it (manual, ~10 min).** Open the link. Look through the Overview and Detections tabs, and the Knowledge
graph tab (try `intranet-server`, `T1078`, `alyssa.hayes`). In the chat, as **tier1**, ask at least five questions of
your own, including two follow-ups that need memory ("…and who owns it?") and one restricted question. Switch
*view as* to **manager** and ask the restricted question again. Take the screenshots for E-1.

In [ ]:
# E.3 Acceptance test: drives the running product end to end (~1 min)
H.acceptance(level="a6")

In [ ]:
# E.4 Save your copilot for A7: writes my_a6.py (your build_kg, kg_context, prompt, answer and memory code).
#     Download it (📁 -> right-click -> Download) or keep it in Drive. A7 starts from it.
H.export_a6()

✍️ **Questions E-1 … E-2**
* **E-1** Screenshots: the Overview tab, the KG explorer on a node of your choice, and the chat answering the
  **same restricted question** as tier1 and as manager.
* **E-2** The acceptance scorecard (explain any failed check). Then one question of your own that the copilot got
  wrong: was it retrieval, memory (the rewrite), or generation? Show the evidence.

---
## Part F — Report (in the handout)
1. Tables: B.2 sizes, B-2 extraction, C-models scoreboard, C-budget sweep, D-memory policies.
2. Answers to A-1 … E-2 (short; numbers from *your* run).
3. Screenshots: 0.1, the ontology (A.2), a KG view (B.2), the product (E-1).
4. **AI-assistant log** as in A4/A5: assistant used; three prompts (worked / failed / fix); one place its code ran
   but was wrong, and how you found out.
5. Submit one PDF: the filled handout followed by this notebook printed with its outputs (*File → Print → Save as PDF*).
6. Keep **my_a6.py** — A7 starts from it.